# rift `nisar2cog` — DPS job runner (bounding-box driven)

Discover NISAR **GSLC provisional** granules over an area of interest with `earthaccess`
(CMR), stage them to `my-public-bucket`, and submit one MAAP DPS job per granule to the
`rift-nisar2cog` OGC process. Start on the **`maap-dps-sandbox`** queue.

Run this in a **MAAP Hub (OGC) workspace** so maap-py v5 is available.

**Prereqs**
- The `rift-nisar2cog` OGC process is deployed (via `.github/workflows/ogc-app-pack.yml`).
- `earthaccess` installed (`pip install earthaccess`) and an Earthdata Login.

In [ ]:
# One-time in a fresh workspace:
# %pip install earthaccess
import earthaccess
import pandas as pd
import datetime, json, os, time
from maap.maap import MAAP

maap = MAAP()

# NISAR provisional collections are auth-gated: you MUST log in to Earthdata or searches
# return 0 results. Run once per workspace.
auth = earthaccess.login(strategy="interactive")

## 1. Area of interest

Thwaites / Pine Island sector. The bounding box `(W, S, E, N)` is set in the search cell
below (§2). Source polygon, for reference:
`POLYGON((-101.9351 -74.7848,-102.4704 -75.4215,-99.852 -75.5511,-99.4242 -74.9087,-101.9351 -74.7848))`

## 2. Search for NISAR GSLC granules (earthaccess / CMR)

NISAR provisional data is discovered via **`earthaccess`** (CMR), not asf-search, and
requires an Earthdata login (done above). We query the GSLC provisional collection by
bounding box + time window.

- `short_name="NISAR_L2_GSLC_PROVISIONAL_V1"` — the provisional GSLC collection
- `bounding_box=(W, S, E, N)` — a numeric tuple (not a WKT string)
- `temporal=(start, end)`

Each result exposes `data_links()` (HTTPS) and `data_links(access="direct")` (`s3://`).

In [ ]:
# AOI bounding box (W, S, E, N) from the Thwaites/PIG polygon.
BBOX = (-102.4704, -75.5511, -99.4242, -74.7848)

# Acquisition window (adjust as needed).
START = "2026-07-08T07:00:00Z"
END   = "2026-07-20T06:59:59Z"

results = earthaccess.search_data(
    short_name="NISAR_L2_GSLC_PROVISIONAL_V1",
    bounding_box=BBOX,
    temporal=(START, END),
)
print(f"{len(results)} granules")
len(results)

In [ ]:
# Peek at the first granule: its links and umm metadata (confirm it's GSLC / 40+5).
if len(results):
    g = results[0]
    print("HTTPS:", g.data_links())
    print("S3   :", g.data_links(access="direct"))
    print()
    print(json.dumps(g["umm"].get("GranuleUR"), indent=2))
else:
    print("No granules — check login, widen the date window, or verify the short_name.")

## 3. Build the granule URL list

Each granule exposes an HTTPS link and an `s3://` link to the `.h5`.

**Auth reality (important):** NISAR provisional granules require Earthdata auth to download —
there is no anonymous URL. The DPS worker's current `run.py` downloads `gslc_url` with plain
`requests` (no creds), so a bare NISAR HTTPS/S3 URL **will fail on the worker**. Two options:

1. **Recommended for now:** stage the granule(s) you want into your `my-public-bucket`
   (`s3://maap-ops-workspace/shared/<username>/...`) from this authenticated session, then
   submit the anonymous-HTTPS form
   `https://maap-ops-workspace.s3.amazonaws.com/shared/<username>/...`. See cell below.
2. **Later:** teach `run.py` to pull directly from ASF S3 using MAAP/ASF temporary
   credentials (`maap.aws.earthdata_s3_credentials("https://nisar.asf.earthdatacloud.nasa.gov/s3credentials")`),
   like MAAP's `nisar_access_subset` reference. Then submit the `s3://sds-n-cumulus-...` URL
   directly. (Tracked as a follow-up.)

We collect both link forms here; the staging cell handles option 1.

In [ ]:
def first_h5(links):
    return next((u for u in (links or []) if u.lower().endswith(".h5")),
                (links[0] if links else None))

granules = []
for g in results:
    https = first_h5(g.data_links())
    s3 = first_h5(g.data_links(access="direct"))
    name = g["umm"].get("GranuleUR")
    granules.append({"name": name, "https": https, "s3": s3})

print(f"{len(granules)} granules")
for gr in granules[:5]:
    print(" ", gr["name"])
    print("     https:", gr["https"])
    print("     s3   :", gr["s3"])

# HTTPS links to the .h5 (NOT yet worker-downloadable without auth — see markdown above).
https_urls = [gr["https"] for gr in granules if gr["https"]]
s3_urls    = [gr["s3"] for gr in granules if gr["s3"]]

### 3a. Stage granules into my-public-bucket (so the worker can fetch them anonymously)

Download the granule(s) here (authenticated) and upload to your `my-public-bucket`, then
build the anonymous-HTTPS URLs that `run.py` can pull without credentials. For a first test,
stage just **one** granule (`granules[:1]`).

In [ ]:
# Username + public-bucket prefix.
USERNAME = maap.profile.account_info()["username"]
PUBLIC_PREFIX = f"shared/{USERNAME}/nisar2cog_staging"           # under s3://maap-ops-workspace/
PUBLIC_HTTPS  = f"https://maap-ops-workspace.s3.amazonaws.com/{PUBLIC_PREFIX}"
LOCAL_STAGE   = os.path.expanduser("~/my-public-bucket/nisar2cog_staging")
os.makedirs(LOCAL_STAGE, exist_ok=True)

N = 1                              # <-- widen once the first job works

# earthaccess downloads with your EDL auth; writing under ~/my-public-bucket syncs to S3.
files = earthaccess.download(results[:N], LOCAL_STAGE)
print("downloaded:", files)

staged_urls = [f"{PUBLIC_HTTPS}/{os.path.basename(f)}" for f in files]
for u in staged_urls:
    print("staged:", u)
staged_urls

## 4. Resolve the deployed OGC process id

`maap.list_algorithms()` returns `{"processes": [{"id":..., "title":...}, ...]}`. Find the
`rift-nisar2cog` entry and grab its `id`.

In [ ]:
resp = maap.list_algorithms()
procs = resp.json().get("processes", []) if resp.status_code == 200 else []
for p in procs:
    print(p.get("id"), "|", p.get("title"))

PROCESS_ID = next((p["id"] for p in procs
                   if "nisar2cog" in str(p.get("id", "")).lower()
                   or "nisar2cog" in str(p.get("title", "")).lower()), None)
print("\nPROCESS_ID =", PROCESS_ID)

## 5. Submit one DPS job per granule

Start with a **single** granule (`kept_urls[:1]`) on `maap-dps-sandbox` to validate before
scaling. `submit_job` returns HTTP 202 with `{"id": ..., "status": "accepted"}`.

In [ ]:
QUEUE = "maap-dps-sandbox"        # 8 GB, 10-min cap. Fall back to maap-dps-worker-16gb/-32gb.
TAG = "nisar2cog-thwaites"
POLS = ""                          # empty = all freq-A pols
AMP_ONLY = "false"

# Use the staged (anonymous-HTTPS) URLs the worker can actually download.
TEST_URLS = staged_urls           # already limited to STAGE = granules[:1]
assert PROCESS_ID, "PROCESS_ID not resolved — is the process deployed?"
assert TEST_URLS, "No staged URLs — run the staging cell (3a) first."

rows = []
for i, url in enumerate(TEST_URLS, start=1):
    inputs = {"gslc_url": url, "pols": POLS, "amp_only": AMP_ONLY}
    r = maap.submit_job(process_id=PROCESS_ID, inputs=inputs,
                        queue=QUEUE, dedup=True, tag=TAG)
    body = r.json() if r.status_code == 202 else {}
    job_id, status = body.get("id"), body.get("status", r.text)
    print(f"[{i}/{len(TEST_URLS)}] {r.status_code} job_id={job_id} status={status}")
    rows.append({"n": i, "gslc_url": url, "job_id": job_id,
                 "submit_status": status, "http": r.status_code,
                 "submit_time": datetime.datetime.now().isoformat()})

submit_df = pd.DataFrame(rows)
out_dir = os.path.expanduser("~/my-public-bucket/dps_submission_results")
os.makedirs(out_dir, exist_ok=True)
stamp = datetime.datetime.now().strftime("%Y%m%d%H%M")
csv_path = f"{out_dir}/nisar2cog_{TAG}_{stamp}.csv"
submit_df.to_csv(csv_path, index=False)
print("saved", csv_path)
submit_df

## 6. Monitor jobs and fetch results

`get_job_status` / `get_job_result` return JSON in maap-py v5. Results land under
`~/my-private-bucket/dps_output/rift-nisar2cog/...`.

In [ ]:
for job_id in [j for j in submit_df["job_id"].tolist() if j]:
    s = maap.get_job_status(job_id)
    st = s.json().get("status") if s.status_code == 200 else s.text
    print(job_id, "->", st)

In [ ]:
# Once a job shows succeeded, inspect its outputs:
SUCCESS_JOB_ID = ""  # paste a job id
if SUCCESS_JOB_ID:
    r = maap.get_job_result(SUCCESS_JOB_ID)
    print(json.dumps(r.json(), indent=2) if r.status_code == 200 else r.text)
    # metrics (OGC v5 only):
    m = maap.get_job_metrics(SUCCESS_JOB_ID)
    if m.status_code == 200:
        print(json.dumps(m.json(), indent=2))